# MedEdge AI: Muse Glimmer server on Colab

Serves Meta's open-weight **Muse Glimmer 30B** (Apache 2.0, image and text input) with llama.cpp, then exposes it to the MedEdge backend through a Cloudflare quick tunnel.

- **Runtime:** Runtime > Change runtime type > **L4 GPU** (24 GB). The 17 GB Q4 model plus the 1.4 GB vision projector fit with room for a 64K context.
- **Sessions end** when idle or at Colab's time limit, and the tunnel URL changes every session. The backend falls back to Muse Spark 1.3 automatically, so the app keeps working when this notebook stops.
- The backend only ever sends **redacted** pages here. Never paste unredacted patient documents into this notebook.

Run all cells top to bottom. The first run takes about 15 minutes (llama.cpp build plus an 18 GB download).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build llama.cpp with CUDA

Muse Glimmer support needs llama.cpp build b10353 or newer, so this builds the latest source. Skipped if the binary already exists in this session.

In [ ]:
%%bash
set -e
if [ ! -x /content/llama.cpp/build/bin/llama-server ]; then
  git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
  cd /content/llama.cpp
  cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release > /dev/null
  cmake --build build --config Release -j"$(nproc)" --target llama-server
fi
/content/llama.cpp/build/bin/llama-server --version

## 2. Download the model and vision projector

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "meta-models/Muse-Glimmer-30B-GGUF"
MODEL = hf_hub_download(REPO, "Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf", local_dir="/content/models")
MMPROJ = hf_hub_download(REPO, "mmproj-Muse-Glimmer-30B-Q4_K_M.gguf", local_dir="/content/models")
print(MODEL)
print(MMPROJ)

## 3. Start the server

Two slots of 32K tokens each. `reasoning_strength` low keeps extraction fast; the backend validates every answer, so extra thinking adds little here.

In [ ]:
import secrets
import subprocess
import time
import urllib.error
import urllib.request

API_KEY = secrets.token_urlsafe(24)
server = subprocess.Popen(
    [
        "/content/llama.cpp/build/bin/llama-server",
        "-m", MODEL,
        "--mmproj", MMPROJ,
        "-a", "muse-glimmer",
        "-ngl", "99",
        "-c", "65536",
        "-np", "2",
        "--host", "127.0.0.1",
        "--port", "8080",
        "--api-key", API_KEY,
        "--chat-template-kwargs", '{"reasoning_strength":"low"}',
    ],
    stdout=open("/content/llama-server.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError("llama-server exited; see /content/llama-server.log")
    try:
        if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
            print("Server ready")
            break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(5)
else:
    raise RuntimeError("Server did not become healthy in 15 minutes; see /content/llama-server.log")

## 4. Open a Cloudflare quick tunnel

In [ ]:
import re

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
    stdout=open("/content/cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
)

TUNNEL_URL = None
for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if match:
        TUNNEL_URL = match.group(0)
        break
    time.sleep(1)
if not TUNNEL_URL:
    raise RuntimeError("No tunnel URL yet; see /content/cloudflared.log")

print("Paste these into your .env, then restart the backend:\n")
print(f"GLIMMER_BASE_URL={TUNNEL_URL}/v1")
print(f"GLIMMER_API_KEY={API_KEY}")
print("ROLE_EXTRACTOR=glimmer:muse-glimmer,meta:muse-spark-1.3")

## 5. Check vision through the tunnel

Sends a small synthetic table image. A healthy server lists the three test names. The tunnel can take up to a minute to start answering.

In [ ]:
import base64
import io
import json

from PIL import Image, ImageDraw

img = Image.new("RGB", (640, 220), "white")
draw = ImageDraw.Draw(img)
for i, row in enumerate(["Haemoglobin   11.2   g/dL", "S. Creatinine  1.4   mg/dL", "Platelets      2.1   lakh/cumm"]):
    draw.text((30, 40 + i * 50), row, fill="black")
buffer = io.BytesIO()
img.save(buffer, "PNG")
data_url = "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()

body = {
    "model": "muse-glimmer",
    "max_tokens": 2048,
    "messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": 'List the test names in this table as JSON: {"tests": [...]}'},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    }],
}
request = urllib.request.Request(
    f"{TUNNEL_URL}/v1/chat/completions",
    data=json.dumps(body).encode(),
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
)
for attempt in range(6):
    try:
        reply = json.load(urllib.request.urlopen(request, timeout=300))
        break
    except urllib.error.URLError as exc:
        print(f"Tunnel not ready ({exc}); retrying")
        time.sleep(10)
print(reply["choices"][0]["message"]["content"])

## 6. Keep this tab open

From your laptop, confirm the backend can reach Glimmer:

```
backend\.venv\Scripts\python scripts\smoke_meta.py --only glimmer
```

Server logs, if something looks wrong:

In [ ]:
!tail -n 30 /content/llama-server.log